# CUDA 설치 확인

In [1]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1+cu121
CUDA 사용 가능 여부: True
현재 GPU 이름: NVIDIA GeForce GTX 1070


In [1]:
import gc
import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm.notebook import tqdm
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split

# GPU 사용을 위한 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")

현재 사용 중인 장치: cuda


# 경로 리스트업

In [4]:
# 경로 설정
base_train_dir = "./Train/"
base_morph_dir = "./Train/morpheme/"

# 01부터 16까지 폴더 이름 생성
user_ids = [f"{i:02d}" for i in range(1, 17)] 

# 실제로 존재하는 폴더인지 확인 (직관적 확인용)
for uid in user_ids:
    p = os.path.join(base_train_dir, uid)
    exists = "존재" if os.path.exists(p) else "미존재"
    print(f"사용자 {uid} 데이터 폴더: {exists}")

사용자 01 데이터 폴더: 존재
사용자 02 데이터 폴더: 존재
사용자 03 데이터 폴더: 존재
사용자 04 데이터 폴더: 존재
사용자 05 데이터 폴더: 존재
사용자 06 데이터 폴더: 존재
사용자 07 데이터 폴더: 존재
사용자 08 데이터 폴더: 존재
사용자 09 데이터 폴더: 존재
사용자 10 데이터 폴더: 존재
사용자 11 데이터 폴더: 존재
사용자 12 데이터 폴더: 존재
사용자 13 데이터 폴더: 존재
사용자 14 데이터 폴더: 존재
사용자 15 데이터 폴더: 존재
사용자 16 데이터 폴더: 존재


# 데이터 로드

In [5]:
X_raw_list = []
y_list = []
user_id_list = []
word_to_idx = {}
idx_counter = 0
TARGET_WORD_COUNT = 100 

for uid in tqdm(user_ids, desc="16명 데이터 통합 중"):
    morph_path = os.path.join(base_morph_dir, uid)
    coord_path = os.path.join(base_train_dir, uid)
    
    # 해당 사용자의 정답 JSON 파일들 가져오기
    morph_files = glob.glob(os.path.join(morph_path, "*_morpheme.json"))
    
    for j_path in morph_files:
        with open(j_path, 'r', encoding='utf-8') as f:
            meta = json.load(f)
        
        # 1. 정보 추출
        word = meta['data'][0]['attributes'][0]['name'] # 정답 단어
        start_t = meta['data'][0]['start']             # 시작 시간
        end_t = meta['data'][0]['end']                 # 종료 시간
        folder_name = meta['metaData']['name'].replace('.mp4', '')
        
        target_coord_dir = os.path.join(coord_path, folder_name)
        if not os.path.exists(target_coord_dir): 
            print("못 찾은 경로:", target_coord_dir)
            continue

        # 2. 단어 라벨링 (Target 개수 제한)
        if word not in word_to_idx:
            if len(word_to_idx) >= TARGET_WORD_COUNT: continue
            word_to_idx[word] = idx_counter
            idx_counter += 1
        
        # 3. 프레임 추출 (30fps 가정)
        json_frames = sorted(glob.glob(os.path.join(target_coord_dir, "*.json")))
        fps = 30
        selected_frames = json_frames[int(start_t*fps) : int(end_t*fps)]
        
        # 4. 프레임별 키포인트 가공 (직관적 정규화)
        sequence = []
        for frame_path in selected_frames:
            with open(frame_path, 'r') as f:
                f_data = json.load(f)
            
            if not f_data['people']:
                print("프레임 없음:", frame_path)
                sequence.append(np.zeros(135))
                continue
            
            person = f_data['people']
            lh = np.array(person['hand_left_keypoints_2d'])  # 63개
            rh = np.array(person['hand_right_keypoints_2d']) # 63개
            pose = np.array(person['pose_keypoints_2d'])     # 75개
            
            # 최종 135차원 조합
            nose_shol_idx = [0, 1, 2, 6, 7, 8, 15, 16, 17]
            combined = np.concatenate([lh, rh, pose[nose_shol_idx]]) 
            sequence.append(combined)
        
        # 유효한 동작만 추가
        if len(sequence) > 5:
            X_raw_list.append(np.array(sequence, dtype=np.float32))
            y_list.append(word_to_idx[word])
            user_id_list.append(int(uid))

# 디스크에 굽기 위해 가변 길이 객체 배열을 처리하는 넘파이 고유 포맷 활용
np.savez_compressed(
    "./Train/raw_sign_dataset.npz", 
    X_raw=np.array(X_raw_list, dtype=object), # 각 요소의 프레임 길이가 다르므로 object 타입
    y=np.array(y_list, dtype=np.int32),
    user_ids=np.array(user_id_list, dtype=np.int32),
    word_to_idx=np.array([word_to_idx]) # 딕셔너리 보존
)
print(f"총 {len(X_raw_list)}개의 샘플 로드 완료!")
print(f"학습할 단어 목록: {list(word_to_idx.keys())}")

16명 데이터 통합 중:   0%|          | 0/16 [00:00<?, ?it/s]

총 7810개의 샘플 로드 완료!
학습할 단어 목록: ['고민', '뻔뻔', '수어', '남아', '눈', '독신', '음료수', '발가락', '슬프다', '자극', '안타깝다', '어색하다', '여아', '외국인', '영아', '신사', '뉴질랜드', '나사렛대학교', '알아서', '장애인', '열아홉번째', '침착', '성실', '학교연혁', '싫어하다', '급하다', '필기시험', '병문안', '검사', '결승전', '낚시터', '낚시대', '당뇨병', '독서', '매표소', '면역', '감기', '배드민턴', '변비', '병명', '보건소', '불면증', '불행', '붕대', '사위', '설사', '성병', '방충', '소화제', '손녀', '손자', '수면제', '수집가', '여행지', '예식장', '올림픽경기', '회복', '첫번째', '운동경기', '입원', '재혼', '진단서', '축구장', '치료', '치료법', '친아들', '퇴원', '한약', '한약방', '빈혈', '화상', '가래떡', '고깃국', '고추', '고추가루', '사골', '배추국', '꽈베기', '벌꿀', '꿀물', '냄비', '찬물', '다과', '지방경찰청장', '된장찌게', '돼지고기', '두부', '딸기', '떡국', '라면', '막걸리', '무', '밥그릇', '밥솥', '보신탕', '부엌', '소불고기', '비빔밥', '사과', '사이다']


# 넘파이 배열 로드

In [2]:
data = np.load("./Train/raw_sign_dataset.npz", allow_pickle=True)
X_raw = data['X_raw']
y = data['y']
sample_user_ids = data['user_ids']
word_to_idx = data['word_to_idx'][0]
idx_to_word = {v: k for k, v in word_to_idx.items()}

print(f"📦 원본 데이터 로드 완료! 총 샘플 수: {len(X_raw)}")
print(f"🏷️ 학습할 단어 개수: {len(word_to_idx)}")

📦 원본 데이터 로드 완료! 총 샘플 수: 7810
🏷️ 학습할 단어 개수: 100


## 사용자 한 명의 데이터를 테스트용으로 사용

In [20]:
TEST_USER_ID = 1

train_indices = np.where(sample_user_ids != TEST_USER_ID)[0]
test_indices = np.where(sample_user_ids == TEST_USER_ID)[0]

X_train_raw, y_train = X_raw[train_indices], y[train_indices]
X_test_raw, y_test = X_raw[test_indices], y[test_indices]

print(len(X_train_raw), len(X_test_raw))

7310 500


## 전체 데이터의 10%를 테스트용으로 사용

In [3]:
# 1. 전체 데이터(X_all, y_all)에서 10%를 순수 테스트 세트(봉인)로 분할
X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw, y, 
    test_size=0.10, 
    random_state=42, 
    stratify=y  # 💡 단어별 정답 비율을 훈련/테스트에 똑같이 골고루 분배
)

# 2. 남은 90% 중에서 다시 10%를 조기 종료 '검증용(Val)' 세트로 분할
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw, y_train_val, 
    test_size=0.10, 
    random_state=42, 
    stratify=y_train_val
)

print("===== 📊 논문 스펙 기준 3분할 셔플 결과 =====")
print(f"📚 Train 세트 (가중치 학습용)      : {len(X_train_raw)}개")
print(f"🧪 Validation 세트 (조기종료 체크용): {len(X_val_raw)}개")
print(f"🏅 Pure Test 세트 (최종 실력 검증용): {len(X_test_raw)}개")
print("==============================================")

===== 📊 논문 스펙 기준 3분할 셔플 결과 =====
📚 Train 세트 (가중치 학습용)      : 6326개
🧪 Validation 세트 (조기종료 체크용): 703개
🏅 Pure Test 세트 (최종 실력 검증용): 781개


# 데이터 전처리

In [4]:
def process_and_pad(raw_sequences, max_len=175):
    processed = []
    WIDTH, HEIGHT = 1920.0, 1080.0
    
    # 135차원 중 X, Y 좌표 인덱스 추출 규칙 (X, Y, Score 순서이므로)
    x_indices = list(range(0, 135, 3))
    y_indices = list(range(1, 135, 3))
    
    for seq in raw_sequences:
        padded_seq = np.zeros((max_len, 135), dtype=np.float32)
        actual_len = min(len(seq), max_len)
        
        # Pre-padding 적용
        # 실시간 웹캠 인식 코드에서 deque 사용으로 0이 앞에 적용됨
        start_idx = max_len - actual_len
        padded_seq[start_idx:] = seq[:actual_len]
        
        # 정규화
        for f in range(start_idx, max_len):
            ref_x = padded_seq[f, 126]  
            ref_y = padded_seq[f, 127]  
            
            if ref_x != 0 and ref_y != 0:
                # 코 기준 상대좌표 스케일링
                padded_seq[f, x_indices] = (padded_seq[f, x_indices] - ref_x) / WIDTH
                padded_seq[f, y_indices] = (padded_seq[f, y_indices] - ref_y) / HEIGHT
                
        processed.append(padded_seq)
    return np.array(processed, dtype=np.float16)

In [5]:
def scipy_stretch_sequence(seq, target_len):
    """시퀀스의 프레임 길이를 target_len으로 강제 복원/축소하는 선형 보간 함수"""
    curr_len, dims = seq.shape
    if curr_len == target_len:
        return seq
    
    # 원래 프레임 축과 새로운 프레임 축 정의 (0 ~ 1 스케일)
    x_old = np.linspace(0, 1, curr_len)
    x_new = np.linspace(0, 1, target_len)
    
    # 1D 선형 보간기 빌드
    f = interp1d(x_old, seq, axis=0, kind='linear', fill_value="edge")
    return f(x_new)

In [6]:
def apply_spatial_augmentation(seq, scale_range=(0.9, 1.1), angle_range=(-8, 8)):
    """
    하나의 수어 시퀀스(Seq_len, 135)에 공간 스케일링과 회전을 동시에 적용하는 함수
    """
    augmented_seq = seq.copy()
    
    # 135차원 중 X, Y 좌표 인덱스 분리
    x_indices = list(range(0, 135, 3))
    y_indices = list(range(1, 135, 3))
    
    # ─── [1] 무작위 하이퍼파라미터 추출 ───
    # 예: 이번 샘플은 1.05배 키우고, 5도 회전시키겠다!
    scale = np.random.uniform(scale_range[0], scale_range[1])
    angle_deg = np.random.uniform(angle_range[0], angle_range[1])
    angle_rad = np.radians(angle_deg)
    
    # 회전 행렬 정의
    cos_a = np.cos(angle_rad)
    sin_a = np.sin(angle_rad)
    
    for f in range(len(augmented_seq)):
        # 패딩 구역(0)인 프레임은 건너뜀
        if not np.any(augmented_seq[f]):
            continue
            
        # 기준점인 코(Nose) 좌표 추출 (정규화 전 원본 좌표 기준일 때)
        ref_x = augmented_seq[f, 126]
        ref_y = augmented_seq[f, 127]
        
        # ─── [2] 코를 원점으로 만들기 (중심축 고정) ───
        X_centered = augmented_seq[f, x_indices] - ref_x
        Y_centered = augmented_seq[f, y_indices] - ref_y
        
        # ─── [3] 공간 비틀기 연산 (스케일링 & 회전 동시 적용) ───
        # ① 스케일링 (체형 변화)
        X_scaled = X_centered * scale
        Y_scaled = Y_centered * scale
        
        # ② 회전 변환 (카메라 각도 변화)
        X_rotated = X_scaled * cos_a - Y_scaled * sin_a
        Y_rotated = X_scaled * sin_a + Y_scaled * cos_a
        
        # ─── [4] 다시 원래 위치로 복원 ───
        augmented_seq[f, x_indices] = X_rotated + ref_x
        augmented_seq[f, y_indices] = Y_rotated + ref_y
        
    return augmented_seq

In [8]:
X_train_augmented_raw = []
y_train_augmented_raw = []

for seq, label in zip(X_train_raw, y_train):
    curr_len = len(seq)
    
    # 1. 원본 속도 그대로 저장
    X_train_augmented_raw.append(seq)
    y_train_augmented_raw.append(label)
    
    # 2. 1.4배 천천히 움직이는 '거북이 버전' 증강 (프레임 수를 1.4배 늘림)
    # 16번 사용자나 11번 사용자의 느린 템포를 모사합니다.
    slow_len = min(int(curr_len * 1.4), 175)
    if slow_len > curr_len:
        slow_seq = scipy_stretch_sequence(np.array(seq), slow_len)
        X_train_augmented_raw.append(slow_seq)
        y_train_augmented_raw.append(label)
        
    # 3. 0.7배 빠르게 휙휙 움직이는 '치타 버전' 증강 (프레임 수를 0.7배로 압축)
    fast_len = max(int(curr_len * 0.7), 15)
    if fast_len < curr_len:
        fast_seq = scipy_stretch_sequence(np.array(seq), fast_len)
        X_train_augmented_raw.append(fast_seq)
        y_train_augmented_raw.append(label)

    # 4. 공간 비틀기
    spatial_seq = apply_spatial_augmentation(np.array(seq))
    X_train_augmented_raw.append(spatial_seq)
    y_train_augmented_raw.append(label)

print(f"🔥 시간 축 증강(Speed Augmentation) 완료!")
print(f"원본 + 느린 버전 + 빠른 버전 결합 후 샘플 수: {len(X_train_augmented_raw)}개")

🔥 시간 축 증강(Speed Augmentation) 완료!
원본 + 느린 버전 + 빠른 버전 결합 후 샘플 수: 25291개


In [9]:
X_train_final = process_and_pad(X_train_augmented_raw).astype(np.float16)
y_train_final = np.array(y_train_augmented_raw, dtype=np.int16)

X_val_final = process_and_pad(X_val_raw, max_len=175).astype(np.float16)
y_val_final = np.array(y_val, dtype=np.int16)

print(f"📊 훈련 데이터 개수: {X_train_final.shape[0]}개")
print(f"📊 검증 데이터 개수: {X_val_final.shape[0]}개")

📊 훈련 데이터 개수: 25291개
📊 검증 데이터 개수: 703개


In [10]:
X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_final, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_final, dtype=torch.long)
y_val_tensor = torch.tensor(y_val_final, dtype=torch.long)

del X_train_final, y_train_final, X_val_final, y_val_final
gc.collect()

216

In [11]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 모델 학습, 평가

In [12]:
class SignLanguageClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(SignLanguageClassifier, self).__init__()
        # batch_first=True: (Batch, Seq, Feature) 순서로 데이터를 받음
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.5, bidirectional=True)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        # x: (Batch, Seq_len, Input_dim)
        out, _ = self.gru(x)
        # 패딩이 섞여 있어도 영상 전체 프레임에서 '수어 동작이 가장 격렬했거나 핵심적인 특징'을 다 긁어모읍니다.
        avg_pool = torch.mean(out, dim=1)   # (Batch, Hidden_dim * 2)
        max_pool, _ = torch.max(out, dim=1) # (Batch, Hidden_dim * 2)
        
        # 둘을 더하거나 결합하여 특징을 강화합니다.
        combined = torch.cat((avg_pool, max_pool), dim=1) # (Batch, Hidden_dim * 4)
        
        return self.fc(combined)

In [14]:
# 하이퍼파라미터 설정
input_size = 135    # (왼손 21*3 + 오른손 21*3 + 포즈 3*3)
hidden_size = 64   
output_dim = len(word_to_idx)
num_layers = 2
NUM_EPOCHS = 120
patience = 20
patience_counter = 0
best_val_loss = float('inf') 
ACCUMULATION_STEPS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SignLanguageClassifier(input_size, hidden_size, output_dim, num_layers).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, (batch_X, batch_y) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
        
        train_loss += (loss.item() * ACCUMULATION_STEPS) * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += batch_y.size(0)
        correct_train += (predicted == batch_y).sum().item()
        
    train_loss /= total_train
    train_acc = (correct_train / total_train) * 100
    
    model.eval()
    with torch.no_grad():
        val_inputs = X_val_tensor.to(device)
        val_labels = y_val_tensor.to(device)
        
        val_outputs = model(val_inputs)
        val_loss = criterion(val_outputs, val_labels).item()
        
        _, val_predicted = torch.max(val_outputs, 1)
        correct_val = (val_predicted == val_labels).sum().item()
        val_acc = (correct_val / val_labels.size(0)) * 100
        
    # 5에폭마다 혹은 1에폭일 때 진행 상황 출력
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:2d}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% "
              f"|| Val Loss: {val_loss:.4f} | Test Acc: {val_acc:.2f}%")
        
    if val_loss < best_val_loss:
        # 전성기 갱신! 검증 Loss가 더 낮아졌다면 가중치 스냅샷 저장
        best_val_loss = val_loss
        patience_counter = 0  # 카운터 초기화
        torch.save({
            'model_state_dict': model.state_dict(),
            'word_to_idx': word_to_idx,
            'hidden_dim': hidden_size,   # 나중에 자동 싱크용 메타데이터
            'num_layers': num_layers
        }, "best_gru_model.pth")
        print(f" ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: {val_loss:.4f})")
    else:
        # 성능 개선이 안 되었다면 참을성 카운터 1 증가
        patience_counter += 1
        print(f" ➡️ ⏳ 개선 없음 (참을성: {patience_counter}/{patience})")
        
        # 지정한 참을성 한계를 넘어서면 학습을 과감히 정지!
        if patience_counter >= patience:
            print(f"\n🛑 [Early Stopping Triggered] {epoch}에폭에서 학습을 조기 종료합니다.")
            print(f"🏆 최저 검증 Loss (Best Val Loss): {best_val_loss:.4f}")
            break

Epoch [ 1/120] | Train Loss: 4.3618 | Train Acc: 2.10% || Val Loss: 3.8924 | Test Acc: 5.97%
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 3.8924)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 3.4105)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 2.9360)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 2.6604)
Epoch [ 5/120] | Train Loss: 2.6099 | Train Acc: 26.36% || Val Loss: 2.2841 | Test Acc: 40.40%
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 2.2841)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 2.0010)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.7197)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.4844)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.3502)
Epoch [10/120] | Train Loss: 1.4923 | Train Acc: 59.42% || Val Loss: 1.3054 | Test Acc: 67.57%
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.3054)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.1656)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 1.0644)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 0.9662)
 ➡️ 👑 Best Model 가중치 저장 완료! (검증 손실: 0.9261)
Epoch [15/120] | Train Loss: 1.0463 | Train Acc: 71.56% || Val Loss: 0.9128 | Test Acc: 75.25%
 ➡️ 👑 

# 모델 불러오기

In [15]:
# 1. 파일 불러오기
save_path = "best_gru_model.pth"
checkpoint = torch.load(save_path)

# 2. 저장된 정보 추출
loaded_word_to_idx = checkpoint['word_to_idx']
idx_to_word = {v: k for k, v in loaded_word_to_idx.items()} # 숫자를 다시 단어로
hidden_size = checkpoint['hidden_dim']
num_layers = checkpoint['num_layers']
output_dim = len(loaded_word_to_idx)

# 3. 모델 초기화 및 가중치 로드
# 주의: SignLanguageClassifier 클래스가 위쪽 셀에 정의되어 있어야 합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_test = SignLanguageClassifier(135, hidden_size, output_dim, num_layers).to(device)
model_test.load_state_dict(checkpoint['model_state_dict'])
model_test.eval() # 반드시 평가 모드로 설정 (Dropout 비활성화)

print(f"✅ 모델 로드 완료! (인식 가능 단어 수: {output_dim}개)")

✅ 모델 로드 완료! (인식 가능 단어 수: 100개)


/tmp/ipykernel_781/2803092405.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path)


# 불러온 모델 테스트

In [16]:
X_test_final = process_and_pad(X_test_raw, max_len=175).astype(np.float16)
y_test_final = np.array(y_test, dtype=np.int16)

X_test_tensor = torch.tensor(X_test_final, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_final, dtype=torch.long)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [19]:
test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward Pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # 통계 누적
        test_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        total_test += batch_y.size(0)
        correct_test += (predicted == batch_y).sum().item()

# 최종 지표 계산
final_test_loss = test_loss / total_test
final_test_acc = (correct_test / total_test) * 100

print("\n==================================================")
print(f"📊 [테스트 결과 스냅샷]")
print(f"✅ 테스트 손실 (Test Loss)  : {final_test_loss:.4f}")
print(f"✅ 테스트 정확도 (Test Acc) : {final_test_acc:.2f}%")
print("==================================================")


📊 [테스트 결과 스냅샷]
✅ 테스트 손실 (Test Loss)  : 0.3500
✅ 테스트 정확도 (Test Acc) : 91.55%
